## Models In LangChain

### init_chat_model
 - Provides a unified, provider-agnostic interface to easily swap backends without refactoring

In [16]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
os.environ['GEMINI_API_KEY'] = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model1  =init_chat_model("groq:qwen/qwen3.8-27b")
groq_model = model1.invoke("What is RAG?")
model2 = init_chat_model(
    "google_genai:gemini-2.5-flash-lite"
)
gemini_model = model2.invoke("What is Capital of France")

print(groq_model.content)
print("="*62)
print(gemini_model.content)

**RAG** stands for **Retrieval-Augmented Generation**.

It is a popular architecture used in Natural Language Processing (NLP) and Large Language Models (LLMs) that combines the strengths of **retrieval-based methods** and **generative models** to produce more accurate, factually grounded, and up-to-date responses.

### How RAG Works
A RAG system typically involves three main components:

1. **Retriever**: 
   - Searches through a designated knowledge base (e.g., a database of documents, web pages, or internal company data) to find relevant passages that answer the user’s query.
   - Uses techniques like vector search (embeddings) or keyword matching.

2. **Augmentation**: 
   - The retrieved relevant information is “injected” or appended to the user’s original prompt.
   - This provides the LLM with fresh, specific context it may not have seen during its training.

3. **Generator**: 
   - The LLM processes the augmented prompt (original question + retrieved context) and generates a na

### Dedicated Provider Classes
- Use when you need provider-specific constructor parameters:

In [17]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
groq_llm = ChatGroq(
    model="qwen/qwen3.8-27b"
)
gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite"
)
groq_response = groq_llm.invoke("what is Token? Explian in 2 lines")
print(groq_response.content)
print("="*45)
gemini_response = gemini_llm.invoke("what is reranker? Explian in 2 lines")
print(gemini_response.content)

A token is a small unit of text (such as a word, part of a word, or a symbol) that large language models process to understand and generate meaningful content.
It serves as the fundamental building block for AI systems, allowing them to break down language into manageable pieces for analysis and output.
A reranker is an AI model that reorders a list of search results or documents to present the most relevant ones first. It achieves this by understanding the nuances of both the query and the documents to assign a more accurate relevance score.


### Standard Parameters & Resilience
- Standard parameters passed into init_chat_model or constructor **kwargs:

In [18]:
model = init_chat_model(
    "groq:qwen/qwen3.8-27b",
    temperature=0.7,      # Randomness (0.0 = deterministic, 1.0 = creative)
    max_tokens=1024,      # Maximum output token limit
    timeout=60,           # Max seconds before request cancellation
    max_retries=10,       # Automatic retry attempts for 429/5xx (default: 6)

)

res = model.invoke("what is capital of germay only name ")
print(res.content)

Berlin


### Invocation Methods (invoke, stream, batch)
- A chat model must be invoked to generate an output. There are three primary invocation methods, each suited to different use cases.

#### 1. invoke() — Synchronous/Complete Call
- Single Message or List of Messages

In [5]:
## Single Message 

response  = model.invoke("What is Crew AI?")
print(response.content)

**CrewAI** is a lightweight, standalone Python framework designed for orchestrating autonomous AI agents. It allows developers to create "crews" of AI agents that can work together to solve complex tasks by collaborating, delegating, and sharing context.

### Key Features:

1. **Multi-Agent Collaboration**:
   - You can define multiple agents, each with a specific **role**, **goal**, and **backstory**.
   - Agents communicate with each other to achieve a shared objective, mimicking how human teams collaborate.

2. **Task Delegation and Coordination**:
   - Tasks can be assigned to specific agents or allowed to be dynamically delegated based on the agents’ capabilities.
   - CrewAI handles the coordination logic, reducing the need for custom orchestration code.

3. **Integration with LLMs**:
   - Works with various large language models (LLMs) such as OpenAI, Anthropic, Google Gemini, and local models via LiteLLM.
   - Supports tool use, allowing agents to interact with external APIs, d

In [6]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore développer des applications.


### 2. stream()
Streaming generates and displays output piece-by-piece in real time, greatly improving response responsiveness.

- Why use it? Faster perceived speed—users don't have to wait for the entire response to finish generating.
- How it works: .stream() returns an iterator yielding text chunks as they arrive.

In [7]:
for chunk in model.stream("What is RAG?"):
    print(chunk.text, end="",flush=True)

**RAG** stands for **Retrieval-Augmented Generation**. It is a technique used to improve the performance of Large Language Models (LLMs) by allowing them to access and use external sources of information (usually a private or specific knowledge base) in addition to their pre-trained knowledge.

### Why is RAG needed?
LLMs have limitations:
1. **Knowledge Cutoff**: They only know what they were trained on up to a certain date.
2. **Hallucinations**: They may generate factually incorrect information if they are unsure or if the information is not in their training data.
3. **Lack of Private/Specialized Data**: They do not have access to your company’s internal documents, proprietary data, or real-time information.

RAG solves these problems by "grounding" the model’s responses in factual, up-to-date, and context-specific information.

### How RAG Works (Step-by-Step)
The RAG pipeline typically involves two main components: **Retrieval** and **Generation**.

1. **Ingestion & Indexing**:
 

In [8]:
# Accumulate complete message
full_message = None
for chunk in model.stream("Hello"):
    full_message = chunk if full_message is None else full_message + chunk

print(full_message.content)

Hello! How can I help you today?


### 3. astream_events() — Semantic Event Streaming

In [9]:
async for event in model.astream_events("Hello", version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].text, end="", flush=True)

Hello! How can I help you today?

### 📦 Batching Requests: `.batch()` vs `.batch_as_completed()`

Send multiple requests in parallel to improve speed and throughput.

- **`.batch()`** — Waits for **all requests** to finish, then returns results in input order.
- **`.batch_as_completed()`** — Returns each result **as soon as it finishes**, so results may arrive out of order.

### ⚡ `max_concurrency`

Controls the **maximum number of requests running in parallel** when using `.batch()` or `.batch_as_completed()`.

In [12]:
queries = [
    "What is AutoGen?",
    "What is RAG?",
    "Why is the sky blue?"
]

responses = model.batch(queries)

for response in responses:
    print(response.content)

**AutoGen** is an open-source programming framework for building **multi-agent AI systems**. It is developed by Microsoft Research and is designed to simplify the process of developing AI applications that involve multiple Large Language Models (LLMs) working together to solve complex tasks.

### Key Features of AutoGen:

1. **Multi-Agent Collaboration**:
   - AutoGen allows multiple AI agents to interact with each other in a conversational or structured manner.
   - Agents can have different roles (e.g., planner, coder, reviewer, user proxy) and collaborate to accomplish tasks more effectively than a single agent could.

2. **Flexibility and Customizability**:
   - Developers can define custom agent behaviors, tools, and conversation patterns.
   - It supports various communication patterns, such as pairwise conversations, group chats, and hierarchical structures.

3. **Integration with LLMs**:
   - AutoGen is model-agnostic, meaning it can work with various LLM providers (e.g., OpenA

In [13]:
for index, response in model.batch_as_completed(queries):
    print(response.content)

**RAG** stands for **Retrieval-Augmented Generation**. It is an AI architecture pattern that combines the strengths of large language models (LLMs) with external knowledge sources to improve the accuracy and relevance of their responses.

### How RAG Works
RAG addresses a common limitation of standard LLMs: they can "hallucinate" (generate false information) or lack up-to-date/domain-specific knowledge because their training data has a cutoff date.

RAG introduces a retrieval step before generation:

1. **Retrieval**: When a user asks a question, the system first searches a predefined knowledge base (e.g., documents, databases, or APIs) for relevant information snippets related to the query.
2. **Augmentation**: These retrieved snippets are added to the user’s original prompt as context.
3. **Generation**: The LLM uses both the user’s question and the retrieved context to generate a more accurate, grounded, and up-to-date answer.

### Key Benefits
- **Reduced Hallucinations**: By groun